# Maritime Demo Deployment
This notebook deploys all Fabric items from the local Git repository to a Fabric workspace.

**Prerequisites:**
- Azure CLI installed and authenticated (`az login`)
- OR you can use interactive device code authentication

**⚠️ IMPORTANT - Git Workflow:**
- This deployment **modifies local files** (GUIDs, workspace IDs) to match your target workspace
- **DO NOT commit these changes to Git** - they are deployment-specific
- The Git repository should remain workspace-agnostic with template values
- After deployment, use `git checkout .` or `git restore .` to revert local changes


In [57]:
%pip install azure-identity requests fabric-cicd -q

Note: you may need to restart the kernel to use updated packages.


In [58]:
# ===== CONFIGURATION =====
# Specify your target Fabric workspace name or ID
TARGET_WORKSPACE_NAME = "testautomatedcreation"  # ← Update this with your workspace name

# Alternative: Use workspace ID directly if you know it
# TARGET_WORKSPACE_ID = "c98a8692-0093-466a-b818-1fe71a28f3b7"  # ← Uncomment and use this instead

In [59]:
import os
import sys
import json
import requests
from pathlib import Path
from azure.identity import DeviceCodeCredential

# 1. Get the local repository path (current notebook's directory)
notebook_dir = Path(__file__).parent if '__file__' in globals() else Path.cwd()
repo_path = str(notebook_dir.absolute())

print(f"📂 Local repository path: {repo_path}")

# 2. Authenticate to Fabric
print("\n🔐 Authenticating to Microsoft Fabric...")
print("💡 TIP: If you get 'UserNotLicensed', you need a Fabric trial or license.")
print("   Visit: https://app.fabric.microsoft.com → Start trial (free 60 days)\n")

try:
    # Use DeviceCodeCredential for interactive authentication
    credential = DeviceCodeCredential()
    fabric_token = credential.get_token("https://api.fabric.microsoft.com/.default").token
    print("✅ Authentication successful!")
except Exception as e:
    print(f"❌ Authentication failed: {e}")
    sys.exit(1)

# Store credential and token for later use
globals()['credential'] = credential
globals()['fabric_token'] = fabric_token
globals()['repo_path'] = repo_path

# Set up headers for API calls
headers = {
    "Authorization": f"Bearer {fabric_token}",
    "Content-Type": "application/json"
}
globals()['headers'] = headers

print("✅ Ready for workspace lookup!")

📂 Local repository path: /Users/rabindori/workspace/aitour2/maritimedemo/fabricdemo

🔐 Authenticating to Microsoft Fabric...
💡 TIP: If you get 'UserNotLicensed', you need a Fabric trial or license.
   Visit: https://app.fabric.microsoft.com → Start trial (free 60 days)

To sign in, use a web browser to open the page https://login.microsoft.com/device and enter the code L8985URKW to authenticate.
✅ Authentication successful!
✅ Ready for workspace lookup!


In [60]:
# 3. Lookup target workspace by name or ID
print("🔍 Looking up target workspace...\n")

# Check if workspace ID or name is configured
if 'TARGET_WORKSPACE_ID' in globals() and TARGET_WORKSPACE_ID:
    # Use direct workspace ID
    target_workspace_id = TARGET_WORKSPACE_ID
    print(f"✅ Using configured workspace ID: {target_workspace_id}")
    
    # Optionally verify it exists and get the name
    response = requests.get(
        f"https://api.fabric.microsoft.com/v1/workspaces/{target_workspace_id}",
        headers=headers
    )
    
    if response.status_code == 200:
        workspace_data = response.json()
        target_workspace_name = workspace_data.get('displayName', 'Unknown')
        print(f"   Workspace name: {target_workspace_name}")
    else:
        print(f"   ⚠️  Could not fetch workspace details (will continue anyway)")
        target_workspace_name = "Unknown"
        
elif 'TARGET_WORKSPACE_NAME' in globals() and TARGET_WORKSPACE_NAME:
    # Search for workspace by name
    target_workspace_name = TARGET_WORKSPACE_NAME
    print(f"🔍 Searching for workspace: '{target_workspace_name}'...")
    
    # List workspaces to find the matching one
    response = requests.get(
        "https://api.fabric.microsoft.com/v1/workspaces",
        headers=headers
    )
    
    if response.status_code == 200:
        workspaces = response.json().get("value", [])
        
        # Find workspace by name (case-insensitive match)
        target_workspace = next(
            (ws for ws in workspaces if ws['displayName'].lower() == target_workspace_name.lower()),
            None
        )
        
        if target_workspace:
            target_workspace_id = target_workspace['id']
            target_workspace_name = target_workspace['displayName']  # Use actual name for correct casing
            print(f"✅ Found workspace: {target_workspace_name}")
            print(f"   ID: {target_workspace_id}")
        else:
            print(f"❌ Workspace '{target_workspace_name}' not found!")
            print(f"\nAvailable workspaces ({len(workspaces)}):")
            for ws in workspaces[:10]:  # Show first 10
                print(f"  • {ws['displayName']}")
            if len(workspaces) > 10:
                print(f"  ... and {len(workspaces) - 10} more")
            sys.exit(1)
    elif response.status_code == 401 and "UserNotLicensed" in response.text:
        print(f"❌ License Required: Your account doesn't have Fabric access.")
        print("\n🔧 Solutions:")
        print("  1. Start a FREE 60-day Fabric trial:")
        print("     → https://app.fabric.microsoft.com (click 'Start trial')")
        print("  2. Use a different Microsoft account with Fabric access")
        print("  3. Contact your admin to assign a Fabric capacity/license")
        sys.exit(1)
    else:
        print(f"❌ Failed to list workspaces: {response.status_code}")
        sys.exit(1)
else:
    print("❌ No workspace configured!")
    print("   Please set TARGET_WORKSPACE_NAME or TARGET_WORKSPACE_ID in the configuration cell")
    sys.exit(1)

print(f"\n🎯 Target workspace: {target_workspace_name}")
print(f"   ID: {target_workspace_id}")

# Store for later cells
globals()['target_workspace_id'] = target_workspace_id
globals()['target_workspace_name'] = target_workspace_name

🔍 Looking up target workspace...

🔍 Searching for workspace: 'testautomatedcreation'...
✅ Found workspace: testautomatedcreation
   ID: c98a8692-0093-466a-b818-1fe71a28f3b7

🎯 Target workspace: testautomatedcreation
   ID: c98a8692-0093-466a-b818-1fe71a28f3b7


In [61]:
# Step 1: List all items in the workspace
print(f"🔍 Checking items in workspace '{target_workspace_name}'")
print(f"   Workspace ID: {target_workspace_id}\n")

response = requests.get(
    f"https://api.fabric.microsoft.com/v1/workspaces/{target_workspace_id}/items",
    headers=headers
)

# If token expired, refresh it
if response.status_code == 401:
    print("🔄 Token expired, refreshing authentication...")
    fabric_token = credential.get_token("https://api.fabric.microsoft.com/.default").token
    headers["Authorization"] = f"Bearer {fabric_token}"
    globals()['fabric_token'] = fabric_token
    globals()['headers'] = headers
    
    # Retry the request
    response = requests.get(
        f"https://api.fabric.microsoft.com/v1/workspaces/{target_workspace_id}/items",
        headers=headers
    )
    print("✅ Token refreshed successfully\n")

if response.status_code == 200:
    all_items = response.json().get("value", [])
    
    # Filter out system items (SQLEndpoint, GraphModel are auto-generated)
    deletable_items = [
        item for item in all_items 
        if item['type'] not in ['SQLEndpoint', 'GraphModel']
    ]
    
    print(f"Found {len(deletable_items)} items that can be deleted:")
    for item in deletable_items:
        print(f"  • {item['type']}: {item['displayName']}")
    
    print(f"\n💡 Run the next cell to DELETE these items")
else:
    print(f"❌ Failed to list items: {response.status_code}")

🔍 Checking items in workspace 'testautomatedcreation'
   Workspace ID: c98a8692-0093-466a-b818-1fe71a28f3b7

Found 15 items that can be deleted:
  • Report: Vessels By Company
  • SemanticModel: maritimeSM
  • Lakehouse: maritimeLH
  • Eventhouse: maritimeEH
  • KQLDatabase: maritimeEH
  • Ontology: maritimeOntologyfromSM
  • Lakehouse: maritimeOntologyfromSM_lh_4bb9953f60c04eef90a68d05ec8ca3b6
  • Notebook: runVessels
  • Notebook: runVesselswithSimulation2
  • Notebook: runVesselswithSimulation
  • Notebook: createOntology
  • Eventstream: maritimeES
  • DataAgent: maritimeDA
  • Reflex: RedAlertActivator
  • Map: vessels_map

💡 Run the next cell to DELETE these items


In [62]:
# Step 2: DELETE ALL ITEMS (handles dependencies with multiple passes)
import time

# Refresh token if needed (prevents 401 errors during deletion)
print("🔄 Refreshing authentication token...")
fabric_token = credential.get_token("https://api.fabric.microsoft.com/.default").token
headers["Authorization"] = f"Bearer {fabric_token}"
globals()['fabric_token'] = fabric_token
globals()['headers'] = headers
print("✅ Token refreshed\n")

# Safety check: Ask for confirmation if workspace has items
if len(deletable_items) > 0:
    print(f"⚠️  WARNING: You are about to DELETE {len(deletable_items)} items from workspace '{target_workspace_name}'!")
    print(f"\n💡 To proceed with deletion, type 'DELETE' and press Enter.")
    print(f"   To cancel, type anything else or just press Enter.\n")
    
    confirmation = input("Confirm deletion: ").strip()
    
    if confirmation != "DELETE":
        print(f"\n❌ Deletion cancelled. No items were deleted.")
        print(f"   Skipping to deployment...")
        # Set a flag to skip the rest of this cell
        globals()['skip_deletion'] = True
    else:
        print(f"\n✅ Confirmation received. Proceeding with deletion...\n")
        globals()['skip_deletion'] = False
else:
    print(f"✅ Workspace is already empty. Skipping deletion...")
    globals()['skip_deletion'] = True

if not globals().get('skip_deletion', False):
    print(f"⚠️  DELETING ALL ITEMS from workspace '{target_workspace_name}'...\n")
    
    # Define deletion order based on dependencies (reverse of deployment order)
    priority_order = {
        'Report': 1,          # Delete reports first (depend on SemanticModel)
        'SemanticModel': 2,   # Then semantic models (depend on Lakehouse)
        'Reflex': 3,          # Then reflex (depends on Ontology)
        'DataAgent': 4,       # Then data agent
        'Map': 5,             # Then map
        'Notebook': 6,        # Then notebooks
        'Eventstream': 7,     # Then eventstream (depends on Eventhouse)
        'Ontology': 8,        # Then ontology (depends on SemanticModel)
        'KQLDatabase': 9,     # Then KQL DB (child of Eventhouse)
        'Eventhouse': 10,     # Then eventhouse
        'Lakehouse': 11       # Finally lakehouse
    }
    
    max_passes = 5  # Increased to 5 passes for stubborn dependencies
    deleted_total = 0
    retry_delay = 2  # Wait 2 seconds between passes to let Fabric settle
    
    for pass_num in range(1, max_passes + 1):
        print(f"\n{'='*60}")
        print(f"🔄 Deletion Pass {pass_num}/{max_passes}")
        print(f"{'='*60}\n")
        
        response = requests.get(
            f"https://api.fabric.microsoft.com/v1/workspaces/{target_workspace_id}/items",
            headers=headers
        )
        
        if response.status_code != 200:
            print(f"❌ Failed to list items: {response.status_code}")
            break
        
        all_items = response.json().get("value", [])
        
        # Filter out system items (SQLEndpoint, GraphModel are auto-generated)
        deletable_items = [
            item for item in all_items 
            if item['type'] not in ['SQLEndpoint', 'GraphModel']
        ]
        
        if not deletable_items:
            print("✅ No more items to delete!")
            break
        
        # Sort by priority order
        deletable_items.sort(key=lambda x: priority_order.get(x['type'], 99))
        
        print(f"📋 Found {len(deletable_items)} items to delete:")
        for item in deletable_items:
            print(f"   • {item['type']}: {item['displayName']}")
        
        print(f"\n🗑️  Deleting items...")
        deleted_count = 0
        failed_items = []
        
        for item in deletable_items:
            delete_url = f"https://api.fabric.microsoft.com/v1/workspaces/{target_workspace_id}/items/{item['id']}"
            
            try:
                delete_response = requests.delete(delete_url, headers=headers)
                
                if delete_response.status_code in [200, 204]:
                    print(f"   ✅ Deleted: {item['displayName']}")
                    deleted_count += 1
                    deleted_total += 1
                elif delete_response.status_code == 404:
                    # Item already deleted (race condition)
                    print(f"   ℹ️  Already gone: {item['displayName']}")
                    deleted_count += 1
                else:
                    error_msg = delete_response.text[:100] if delete_response.text else str(delete_response.status_code)
                    print(f"   ⏳ Blocked: {item['displayName']} (status {delete_response.status_code})")
                    failed_items.append(item)
            except Exception as e:
                print(f"   ❌ Error deleting {item['displayName']}: {str(e)[:50]}")
                failed_items.append(item)
        
        print(f"\n📊 Pass {pass_num} Summary:")
        print(f"   ✅ Deleted: {deleted_count}")
        print(f"   ⏳ Remaining: {len(failed_items)}")
        
        # If no progress, stop early
        if deleted_count == 0 and len(failed_items) > 0:
            print(f"\n⚠️  No progress made. Remaining items may have blocking dependencies:")
            for item in failed_items:
                print(f"      • {item['type']}: {item['displayName']}")
            
            if pass_num < max_passes:
                print(f"\n💡 You can:")
                print(f"   1. Run this cell again to retry")
                print(f"   2. Delete remaining items manually in Fabric portal")
            break
        
        # Small delay between passes to let Fabric catch up
        if pass_num < max_passes and len(failed_items) > 0:
            print(f"\n⏱️  Waiting {retry_delay} seconds before next pass...")
            time.sleep(retry_delay)
    
    print(f"\n{'='*60}")
    print(f"📊 FINAL SUMMARY")
    print(f"{'='*60}")
    print(f"✅ Total deleted: {deleted_total}")
    
    # Check final state
    response = requests.get(
        f"https://api.fabric.microsoft.com/v1/workspaces/{target_workspace_id}/items",
        headers=headers
    )
    
    if response.status_code == 200:
        remaining = [
            item for item in response.json().get("value", [])
            if item['type'] not in ['SQLEndpoint', 'GraphModel']
        ]
        
        if remaining:
            print(f"⚠️  {len(remaining)} items still remain:")
            for item in remaining:
                print(f"   • {item['type']}: {item['displayName']}")
            print(f"\n💡 Options to proceed:")
            print(f"   1. Run this cell again - sometimes multiple attempts work")
            print(f"   2. Delete remaining items manually in Fabric portal")
            print(f"   3. Continue with deployment - some items may redeploy successfully")
        else:
            print(f"✅ Workspace is completely clean!")
            print(f"\n🚀 Ready for fresh deployment!")
            
            # CONDITIONAL COOLDOWN: Only wait if items were actually deleted
            if deleted_total > 0:
                print(f"\n⏱️  COOLDOWN: Waiting 10 seconds for Fabric to process deletions...")
                print(f"   (Prevents 'item not available yet' errors)")
                for remaining_seconds in range(10, 0, -2):
                    print(f"   ⏳ {remaining_seconds} seconds remaining...")
                    time.sleep(2)
                print(f"   ✅ Cooldown complete - workspace ready for deployment!")
            else:
                print(f"   ℹ️  No deletions performed - skipping cooldown")
                print(f"   🚀 Workspace ready for immediate deployment!")
    else:
        print(f"❌ Failed to check final state: {response.status_code}")
else:
    print(f"\nℹ️  Skipping deletion - proceeding directly to deployment...")

🔄 Refreshing authentication token...
✅ Token refreshed

⚠️  WARNING: You are about to DELETE 15 items from workspace 'testautomatedcreation'!

💡 To proceed with deletion, type 'DELETE' and press Enter.
   To cancel, type anything else or just press Enter.


✅ Confirmation received. Proceeding with deletion...

⚠️  DELETING ALL ITEMS from workspace 'testautomatedcreation'...


🔄 Deletion Pass 1/5

📋 Found 15 items to delete:
   • Report: Vessels By Company
   • SemanticModel: maritimeSM
   • Reflex: RedAlertActivator
   • DataAgent: maritimeDA
   • Map: vessels_map
   • Notebook: runVessels
   • Notebook: runVesselswithSimulation2
   • Notebook: runVesselswithSimulation
   • Notebook: createOntology
   • Eventstream: maritimeES
   • Ontology: maritimeOntologyfromSM
   • KQLDatabase: maritimeEH
   • Eventhouse: maritimeEH
   • Lakehouse: maritimeLH
   • Lakehouse: maritimeOntologyfromSM_lh_4bb9953f60c04eef90a68d05ec8ca3b6

🗑️  Deleting items...
   ✅ Deleted: Vessels By Company
   ✅ Delet

In [63]:
# Enable required feature flags for selective deployment
from fabric_cicd import append_feature_flag

# Enable experimental features for items_to_include
append_feature_flag("enable_experimental_features")
append_feature_flag("enable_items_to_include")

print("✅ Feature flags enabled for phased deployment")

✅ Feature flags enabled for phased deployment


In [ ]:
from fabric_cicd import FabricWorkspace, publish_all_items
import os
import time
import requests
import re
import json

# Refresh authentication token before starting deployment
print("🔄 Refreshing authentication token before deployment...")
fabric_token = credential.get_token("https://api.fabric.microsoft.com/.default").token
headers["Authorization"] = f"Bearer {fabric_token}"
globals()['fabric_token'] = fabric_token
globals()['headers'] = headers
print("✅ Token refreshed\n")

# 5. Deploy all items with dependency-aware phasing
print(f"\n🚀 Starting phased deployment to workspace '{target_workspace_name}'...")
print(f"   Source: {repo_path}")

# Helper function to verify items are deployed
def verify_items_deployed(expected_names, item_type, max_retries=5):
    """Verify that items are actually deployed and visible in the workspace"""
    for attempt in range(1, max_retries + 1):
        response = requests.get(
            f"https://api.fabric.microsoft.com/v1/workspaces/{target_workspace_id}/items",
            headers=headers
        )
        
        if response.status_code == 200:
            items = response.json().get("value", [])
            deployed_names = [i['displayName'] for i in items if i['type'] == item_type]
            
            # Check if all expected items are present
            missing = [name for name in expected_names if name not in deployed_names]
            
            if not missing:
                print(f"   ✅ Verified: All {item_type} items are deployed")
                return True
            else:
                if attempt < max_retries:
                    print(f"   ⏳ Attempt {attempt}/{max_retries}: Waiting for {missing} to appear...")
                    time.sleep(5)
                else:
                    print(f"   ⚠️  Warning: Items not showing up: {missing}")
                    return False
        else:
            print(f"   ⚠️  API check failed: {response.status_code}")
            return False
    
    return False

# Helper function to upload files to Lakehouse Files section
def upload_to_lakehouse(lakehouse_id, local_file_path, target_path):
    """Upload a file to the Lakehouse Files section using OneLake API"""
    try:
        # Get a storage token for OneLake operations
        storage_token = credential.get_token("https://storage.azure.com/.default").token
        
        # Read the file content
        with open(local_file_path, 'rb') as f:
            file_content = f.read()
        
        # OneLake path: /workspaces/{workspaceId}/items/{lakehouseId}/Files/{path}
        onelake_url = f"https://onelake.dfs.fabric.microsoft.com/{target_workspace_id}/{lakehouse_id}/Files/{target_path}"
        
        # Use Azure Storage Data Lake Gen2 REST API
        upload_headers = {
            "Authorization": f"Bearer {storage_token}",
            "x-ms-version": "2021-06-08",
            "Content-Type": "application/octet-stream"
        }
        
        # Create the file (PUT with resource=file)
        create_response = requests.put(
            f"{onelake_url}?resource=file",
            headers=upload_headers
        )
        
        if create_response.status_code not in [201, 409]:  # 409 = file already exists, which is ok
            print(f"   ⚠️  Failed to create file: {create_response.status_code} - {create_response.text}")
            return False
        
        # Upload the file content (PATCH to append data)
        patch_headers = upload_headers.copy()
        patch_headers["Content-Length"] = str(len(file_content))
        
        patch_response = requests.patch(
            f"{onelake_url}?action=append&position=0",
            headers=patch_headers,
            data=file_content
        )
        
        if patch_response.status_code != 202:
            print(f"   ⚠️  Failed to append data: {patch_response.status_code} - {patch_response.text}")
            return False
        
        # Flush the data (PATCH with action=flush)
        flush_headers = upload_headers.copy()
        flush_headers["Content-Length"] = "0"
        
        flush_response = requests.patch(
            f"{onelake_url}?action=flush&position={len(file_content)}",
            headers=flush_headers
        )
        
        if flush_response.status_code not in [200, 201]:
            print(f"   ⚠️  Failed to flush file: {flush_response.status_code} - {flush_response.text}")
            return False
        
        print(f"   ✅ Uploaded: {target_path}")
        return True
        
    except Exception as e:
        print(f"   ❌ Error uploading {target_path}: {str(e)}")
        return False



# Initialize the Workspace configuration object with ALL item types
target_workspace_obj = FabricWorkspace(
    workspace_id=target_workspace_id,
    repository_directory=repo_path,
    token_credential=credential,
    # IMPORTANT: Include all item types present in the repo
    item_type_in_scope=[
        "Lakehouse",
        "Eventhouse",
        "KQLDatabase",
        "Notebook",
        "SemanticModel",
        "Report",
        "Reflex",
        "Eventstream",
        "DataAgent",
        "Ontology",
        "Map"
    ]
)

try:
    # Phase 1: Deploy data foundation (Lakehouse, Eventhouse + KQL Database)
    print("\n📋 Phase 1: Deploying data foundation (Lakehouse, Eventhouse + KQL Database)...")
    print("   ℹ️  Deploying ONLY Lakehouse and Eventhouse (with children)")
    
    # Use exclusion regex to exclude everything EXCEPT maritimeLH and maritimeEH
    # Pattern: exclude anything that's NOT maritimeLH or maritimeEH
    publish_all_items(
        target_workspace_obj,
        item_name_exclude_regex=r"^(?!maritimeLH$|maritimeEH$).*"
    )
    print("✅ Data foundation deployment completed")
    
    # VERIFY: Check that Lakehouse and Eventhouse are deployed
    print("\n🔍 Verifying Phase 1 deployment...")
    verify_items_deployed(["maritimeLH"], "Lakehouse")
    verify_items_deployed(["maritimeEH"], "Eventhouse")
    
    # Wait a bit for provisioning
    print("⏱️  Waiting 5 seconds...")
    time.sleep(5)
    
    # Upload GeoJSON files to Lakehouse
    print("\n📁 Uploading GeoJSON files to Lakehouse...")
    
    # Get the lakehouse ID
    response = requests.get(
        f"https://api.fabric.microsoft.com/v1/workspaces/{target_workspace_id}/items",
        headers=headers
    )
    
    if response.status_code == 200:
        items = response.json().get("value", [])
        lakehouse = next((i for i in items if i['type'] == 'Lakehouse' and i['displayName'] == 'maritimeLH'), None)
        
        if lakehouse:
            lakehouse_id = lakehouse['id']
            
            # Upload GeoJSON files
            resources_folder = f"{repo_path}/resources"
            geojson_files = [
                "HormuzShippingCorridor.geojson",
                "hormuz_risk_zone.geojson"
            ]
            
            for geojson_file in geojson_files:
                local_path = f"{resources_folder}/{geojson_file}"
                if os.path.exists(local_path):
                    upload_to_lakehouse(lakehouse_id, local_path, geojson_file)
    
    # Post-Phase 1: Update createOntology notebook with Kusto cluster URI
    print("\n🔧 Post-Phase 1: Updating createOntology notebook with Kusto cluster URI...")
    
    # Get the Eventhouse and KQL Database
    response = requests.get(
        f"https://api.fabric.microsoft.com/v1/workspaces/{target_workspace_id}/items",
        headers=headers
    )
    
    if response.status_code == 200:
        items = response.json().get("value", [])
        eventhouse = next((i for i in items if i['type'] == 'Eventhouse' and i['displayName'] == 'maritimeEH'), None)
        kql_db = next((i for i in items if i['type'] == 'KQLDatabase' and i['displayName'] == 'maritimeEH'), None)
        
        if eventhouse and kql_db:
            kql_db_id = kql_db['id']
            
            # Get KQL Database properties which include the query URI
            db_props_url = f"https://api.fabric.microsoft.com/v1/workspaces/{target_workspace_id}/kqlDatabases/{kql_db_id}"
            db_response = requests.get(db_props_url, headers=headers)
            
            if db_response.status_code == 200:
                db_data = db_response.json()
                properties = db_data.get('properties', {})
                kusto_uri = properties.get('queryServiceUri')
                
                if kusto_uri:
                    print(f"   ✅ Retrieved Kusto cluster URI: {kusto_uri}")
                    
                    # Update the createOntology notebook
                    notebook_path = f"{repo_path}/createOntology.Notebook/notebook-content.py"
                    
                    if os.path.exists(notebook_path):
                        with open(notebook_path, 'r') as f:
                            content = f.read()
                        
                        # 1. Replace the Kusto cluster configuration
                        updated_content = re.sub(
                            r'kusto_cluster = "https://[^"]+\.kusto\.fabric\.microsoft\.com"',
                            f'kusto_cluster = "{kusto_uri}"',
                            content
                        )
                        
                        # 2. Update lakehouse metadata (workspace ID and lakehouse ID)
                        # This ensures the notebook writes to the NEW workspace's lakehouse
                        updated_content = re.sub(
                            r'"default_lakehouse_workspace_id":\s*"[^"]*"',
                            f'"default_lakehouse_workspace_id": "{target_workspace_id}"',
                            updated_content
                        )
                        
                        updated_content = re.sub(
                            r'"default_lakehouse":\s*"[^"]*"',
                            f'"default_lakehouse": "{lakehouse_id}"',
                            updated_content
                        )
                        
                        # Update known_lakehouses array to only contain the NEW lakehouse
                        updated_content = re.sub(
                            r'"known_lakehouses":\s*\[[\s\n]*\{[\s\n]*"id":\s*"[^"]*"[\s\n]*\}[\s\n]*\]',
                            f'"known_lakehouses": [\n# META         {{\n# META           "id": "{lakehouse_id}"\n# META         }}\n# META       ]',
                            updated_content
                        )
                        
                        if updated_content != content:
                            with open(notebook_path, 'w') as f:
                                f.write(updated_content)
                            print(f"   ✅ Updated createOntology notebook:")
                            print(f"      - Kusto cluster: {kusto_uri}")
                            print(f"      - Lakehouse workspace: {target_workspace_id}")
                            print(f"      - Lakehouse ID: {lakehouse_id}")
                        else:
                            print(f"   ℹ️  createOntology notebook already has correct configuration")
                    else:
                        print(f"   ⚠️  createOntology notebook not found: {notebook_path}")
                else:
                    print(f"   ⚠️  Kusto cluster URI not found in KQL Database properties")
            else:
                print(f"   ⚠️  Failed to get KQL Database properties: {db_response.status_code}")
        else:
            print(f"   ⚠️  Eventhouse or KQL Database not found")
    
    # Pre-Phase 2: Update SemanticModel lakehouse reference
    print("\n🔧 Pre-Phase 2: Updating SemanticModel lakehouse reference...")
    
    # Get the deployed lakehouse ID
    response = requests.get(
        f"https://api.fabric.microsoft.com/v1/workspaces/{target_workspace_id}/items",
        headers=headers
    )
    
    if response.status_code == 200:
        items = response.json().get("value", [])
        lakehouse = next((i for i in items if i['type'] == 'Lakehouse' and i['displayName'] == 'maritimeLH'), None)
        
        if lakehouse:
            lakehouse_id = lakehouse['id']
            print(f"   📍 Found lakehouse: {lakehouse_id}")
            
            # Update the expressions.tmdl file with new lakehouse ID
            expressions_file = f"{repo_path}/maritimeSM.SemanticModel/definition/expressions.tmdl"
            
            if os.path.exists(expressions_file):
                with open(expressions_file, 'r') as f:
                    content = f.read()
                
                # Pattern to match the Source line with workspace and lakehouse IDs
                # Note: Includes [HierarchicalNavigation=true] parameter
                pattern = r'Source = AzureStorage\.DataLake\("https://onelake\.dfs\.fabric\.microsoft\.com/[^/]+/[^"]+", \[HierarchicalNavigation=true\]\)'
                replacement = f'Source = AzureStorage.DataLake("https://onelake.dfs.fabric.microsoft.com/{target_workspace_id}/{lakehouse_id}", [HierarchicalNavigation=true])'
                
                updated_content = re.sub(pattern, replacement, content)
                
                if updated_content != content:
                    with open(expressions_file, 'w') as f:
                        f.write(updated_content)
                    print(f"   ✅ Updated expressions.tmdl with new lakehouse reference")
                else:
                    print(f"   ⚠️  Warning: File content unchanged - pattern may not have matched")
    
    # Phase 2: Deploy SemanticModel + Report
    print("\n📋 Phase 2: Deploying SemanticModel + Report...")
    try:
        # Exclude everything EXCEPT maritimeSM and Vessels By Company
        publish_all_items(
            target_workspace_obj,
            item_name_exclude_regex=r"^(maritimeLH|maritimeEH|maritimeOntologyfromSM|RedAlertActivator|createOntology|runVessels|runVesselswithSimulation|runVesselswithSimulation2|maritimeES|maritimeDA|vessels_map)$"
        )
        print("✅ SemanticModel + Report deployment completed")
    except Exception as sm_error:
        print(f"\n❌ Phase 2 deployment FAILED:")
        print(f"   Error: {str(sm_error)}")
        import traceback
        traceback.print_exc()
        raise sm_error
    
    # VERIFY: Check that SemanticModel is deployed and ready
    print("\n🔍 Verifying Phase 2 deployment...")
    if not verify_items_deployed(["maritimeSM"], "SemanticModel", max_retries=8):
        raise Exception("SemanticModel deployment verification failed - cannot proceed to Ontology")
    
    # Wait for SemanticModel metadata processing (verification already waited up to 40s)
    print("⏱️  Waiting 10 seconds for metadata processing...")
    time.sleep(10)
    
    # Pre-Phase 3: Update Ontology resource links to point to the deployed report
    print("\n🔗 Pre-Phase 3: Updating Ontology resource links...")
    
    # Get the deployed report ID
    response = requests.get(
        f"https://api.fabric.microsoft.com/v1/workspaces/{target_workspace_id}/items",
        headers=headers
    )
    
    if response.status_code == 200:
        items = response.json().get("value", [])
        report = next((i for i in items if i['type'] == 'Report' and i['displayName'] == 'Vessels By Company'), None)
        
        if report:
            report_id = report['id']
            print(f"   📊 Found report: {report_id}")
            
            # Update the ResourceLinks definition
            resource_links_file = f"{repo_path}/maritimeOntologyfromSM.Ontology/EntityTypes/98576905401063/ResourceLinks/definition.json"
            
            if os.path.exists(resource_links_file):
                # Create the updated resource links
                updated_links = {
                    "$schema": "https://developer.microsoft.com/json-schemas/fabric/item/ontology/resourceLinks/1.0.0/schema.json",
                    "resourceLinks": [
                        {
                            "type": "PowerBIReport",
                            "workspaceId": target_workspace_id,
                            "itemId": report_id
                        }
                    ]
                }
                
                with open(resource_links_file, 'w') as f:
                    json.dump(updated_links, f, indent=2)
                
                print(f"   ✅ Updated resource links to point to Vessels By Company report")
            else:
                print(f"   ⚠️  ResourceLinks file not found: {resource_links_file}")
        else:
            print("   ⚠️  Vessels By Company report not found in workspace")
    else:
        print(f"   ⚠️  Failed to get workspace items: {response.status_code}")
    
    # Phase 3: Deploy Ontology (depends on SemanticModel)
    print("\n📋 Phase 3: Deploying Ontology (with correct resource links)...")
    try:
        # Exclude everything EXCEPT Ontology
        publish_all_items(
            target_workspace_obj,
            item_name_exclude_regex=r"^(?!maritimeOntologyfromSM$).*"
        )
        print("✅ Ontology deployment completed")
    except Exception as ontology_error:
        print(f"\n⚠️  Ontology deployment failed on first attempt: {ontology_error}")
        print("   Retrying after additional wait...")
        time.sleep(15)
        
        # Retry once
        publish_all_items(
            target_workspace_obj,
            item_name_exclude_regex=r"^(?!maritimeOntologyfromSM$).*"
        )
        print("✅ Ontology deployed on retry")
    
    # Verify Ontology
    print("\n🔍 Verifying Phase 3 deployment...")
    verify_items_deployed(["maritimeOntologyfromSM"], "Ontology")
    
    # Phase 4: Deploy Notebooks, Eventstream, DataAgent
    print("\n📋 Phase 4: Deploying Notebooks, Eventstream, DataAgent...")
    # Exclude already-deployed items and Map/Reflex (deploy those last)
    publish_all_items(
        target_workspace_obj,
        item_name_exclude_regex=r"^(maritimeLH|maritimeEH|maritimeSM|Vessels By Company|maritimeOntologyfromSM|RedAlertActivator|vessels_map)$"
    )
    print("✅ Supporting items deployment completed")
    
    # Post-Phase 4: Event Hub configuration is manual (see README.md)
    print("   ℹ️  Event Hub configuration required - see README.md for steps")
    
    # Phase 5: Deploy Reflex (depends on Ontology)
    print("\n📋 Phase 5: Deploying Reflex...")
    publish_all_items(
        target_workspace_obj,
        item_name_exclude_regex=r"^(?!RedAlertActivator$).*"
    )
    print("✅ Reflex deployment completed")
    
    # Phase 6: Deploy Map (depends on Ontology)
    print("\n📋 Phase 6: Deploying Map (depends on Ontology)...")
    try:
        # Check if vessels_map folder exists
        map_folder = f"{repo_path}/vessels_map.Map"
        if os.path.exists(map_folder):
            publish_all_items(
                target_workspace_obj,
                item_name_exclude_regex=r"^(?!vessels_map$).*"
            )
            print("✅ Map deployment completed")
        else:
            print("   ℹ️  Map folder not found - skipping")
    except Exception as map_error:
        print(f"\n⚠️  Map deployment failed: {map_error}")
        print("   💡 Map items may need manual creation in Fabric portal")
        # Don't raise - Map failure is non-critical, continue with verification
    
    # Final verification
    print("\n🔍 Final verification - checking all items...")
    response = requests.get(
        f"https://api.fabric.microsoft.com/v1/workspaces/{target_workspace_id}/items",
        headers=headers
    )
    
    if response.status_code == 200:
        final_items = response.json().get("value", [])
        print(f"\n✅ Deployment complete! Total items in workspace: {len(final_items)}")
        
        by_type = {}
        for item in final_items:
            item_type = item['type']
            by_type[item_type] = by_type.get(item_type, 0) + 1
        
        print("\n📊 Items by type:")
        for item_type, count in sorted(by_type.items()):
            print(f"   {item_type}: {count}")
    
    print(f"\n✅ Maritime Demo deployed successfully! All items with dependencies resolved.")
    print(f"\n📊 View your workspace: https://app.fabric.microsoft.com/groups/{target_workspace_id}")
    
    print(f"\n{'='*70}")
    print(f"🚀 NEXT STEPS - Initialize Data")
    print(f"{'='*70}")
    print(f"\n1️⃣  Run the 'createOntology' notebook to create lakehouse tables:")
    print(f"   • Open createOntology notebook in Fabric portal")
    print(f"   • Run all cells to create: companies, vessels, cargo, policies tables")
    print(f"   • This populates the lakehouse with sample data")
    print(f"\n2️⃣  Refresh the SemanticModel to register metadata:")
    print(f"   • Open 'maritimeSM' SemanticModel in Fabric portal")
    print(f"   • Click 'Refresh now' button in the toolbar")
    print(f"   • Wait ~30 seconds for refresh to complete")
    print(f"   • This validates schema and registers lakehouse connection")
    print(f"\n3️⃣  Verify the Power BI report:")
    print(f"   • Open 'Vessels By Company' report")
    print(f"   • All visuals should now display data")
    print(f"\n4️⃣  Configure Event Hub for real-time vessel data:")
    print(f"   • See README.md 'Post-Deployment Configuration' section")
    print(f"   • Update runVessels notebooks with Event Hub connection")
    print(f"\n💡 See README.md 'Initialize Ontology Tables' for detailed instructions")
    print(f"{'='*70}\n")
    
except Exception as e:
    print(f"\n❌ Deployment failed: {str(e)}")
    print(f"\n💡 Troubleshooting steps:")
    print(f"   1. Run the verification cell to check what deployed successfully")
    print(f"   2. Check the Fabric portal to see if items are visible")
    print(f"   3. Review error details above for specific issues")
    print(f"   4. Re-run this cell to retry failed items (it's safe to re-run)")
    raise

[info]   11:39:25 - 
[info]   11:39:25 - ####################################################################################################
[info]   11:39:25 - ########## Validating Parameter File ###############################################################
[info]   11:39:25 - ####################################################################################################
[info]   11:39:25 - 
[warn]   11:39:25 - Parameter file not found with path: /Users/rabindori/workspace/aitour2/maritimedemo/fabricdemo/parameter.yml
[warn]   11:39:25 - Validation terminated: not found


🔄 Refreshing authentication token before deployment...
✅ Token refreshed


🚀 Starting phased deployment to workspace 'testautomatedcreation'...
   Source: /Users/rabindori/workspace/aitour2/maritimedemo/fabricdemo

📋 Phase 1: Deploying data foundation (Lakehouse, Eventhouse + KQL Database)...
   ℹ️  Deploying ONLY Lakehouse and Eventhouse (with children)


[info]   11:39:28 - 
[info]   11:39:28 - ####################################################################################################
[info]   11:39:28 - ########## Publishing Workspace Folders ############################################################
[info]   11:39:28 - ####################################################################################################
[info]   11:39:28 - 
[info]   11:39:28 - Publishing Workspace Folders
         11:39:28 - Published
[warn]   11:39:30 - Using item_name_exclude_regex is risky as it can prevent needed dependencies from being deployed.  Use at your own risk.
[info]   11:39:30 - 
[info]   11:39:30 - ####################################################################################################
[info]   11:39:30 - ########## Publishing Item 4/29: Lakehouse #########################################################
[info]   11:39:30 - ############################################################################################

✅ Data foundation deployment completed

🔍 Verifying Phase 1 deployment...
   ✅ Verified: All Lakehouse items are deployed
   ✅ Verified: All Eventhouse items are deployed
⏱️  Waiting 5 seconds...

📁 Uploading GeoJSON files to Lakehouse...
   ✅ Uploaded: HormuzShippingCorridor.geojson
   ✅ Uploaded: hormuz_risk_zone.geojson

🔧 Post-Phase 1: Updating createOntology notebook with Kusto cluster URI...
   ✅ Retrieved Kusto cluster URI: https://trd-q49aej1yhzpkrrk4cq.z3.kusto.fabric.microsoft.com
   ✅ Updated createOntology notebook:
      - Kusto cluster: https://trd-q49aej1yhzpkrrk4cq.z3.kusto.fabric.microsoft.com
      - Lakehouse workspace: c98a8692-0093-466a-b818-1fe71a28f3b7
      - Lakehouse ID: 462a13ab-b950-4114-bef2-01ca498b2d4d

🔧 Pre-Phase 2: Updating SemanticModel lakehouse reference...
   📍 Found lakehouse: 462a13ab-b950-4114-bef2-01ca498b2d4d
   ✅ Updated expressions.tmdl with new lakehouse reference

📋 Phase 2: Deploying SemanticModel + Report...


[info]   11:40:53 - 
[info]   11:40:53 - ####################################################################################################
[info]   11:40:53 - ########## Publishing Workspace Folders ############################################################
[info]   11:40:53 - ####################################################################################################
[info]   11:40:53 - 
[info]   11:40:53 - Publishing Workspace Folders
         11:40:53 - Published
[warn]   11:40:54 - Using item_name_exclude_regex is risky as it can prevent needed dependencies from being deployed.  Use at your own risk.
[info]   11:40:54 - 
[info]   11:40:54 - ####################################################################################################
[info]   11:40:54 - ########## Publishing Item 4/29: Lakehouse #########################################################
[info]   11:40:54 - ############################################################################################

...

✅ Supporting items deployment completed
   ℹ️  Event Hub configuration required - see README.md for steps

📋 Phase 5: Deploying Reflex...


[info]   11:43:45 - 
[info]   11:43:45 - ####################################################################################################
[info]   11:43:45 - ########## Publishing Workspace Folders ############################################################
[info]   11:43:45 - ####################################################################################################
[info]   11:43:45 - 
[info]   11:43:45 - Publishing Workspace Folders
         11:43:45 - Published
[warn]   11:43:47 - Using item_name_exclude_regex is risky as it can prevent needed dependencies from being deployed.  Use at your own risk.
[info]   11:43:47 - 
[info]   11:43:47 - ####################################################################################################
[info]   11:43:47 - ########## Publishing Item 4/29: Lakehouse #########################################################
[info]   11:43:47 - ############################################################################################

✅ Reflex deployment completed

📋 Phase 6: Deploying Map (depends on Ontology)...


[info]   11:44:00 - 
[info]   11:44:00 - ####################################################################################################
[info]   11:44:00 - ########## Publishing Workspace Folders ############################################################
[info]   11:44:00 - ####################################################################################################
[info]   11:44:00 - 
[info]   11:44:00 - Publishing Workspace Folders
         11:44:00 - Published
[warn]   11:44:02 - Using item_name_exclude_regex is risky as it can prevent needed dependencies from being deployed.  Use at your own risk.
[info]   11:44:02 - 
[info]   11:44:02 - ####################################################################################################
[info]   11:44:02 - ########## Publishing Item 4/29: Lakehouse #########################################################
[info]   11:44:02 - ############################################################################################

✅ Map deployment completed

🔍 Final verification - checking all items...

✅ Deployment complete! Total items in workspace: 18

📊 Items by type:
   DataAgent: 1
   Eventhouse: 1
   Eventstream: 1
   GraphModel: 1
   KQLDatabase: 1
   Lakehouse: 2
   Map: 1
   Notebook: 4
   Ontology: 1
   Reflex: 1
   Report: 1
   SQLEndpoint: 2
   SemanticModel: 1

✅ Maritime Demo deployed successfully! All items with dependencies resolved.

📊 View your workspace: https://app.fabric.microsoft.com/groups/c98a8692-0093-466a-b818-1fe71a28f3b7


In [65]:
# ⚠️ RUN THIS CELL AFTER DEPLOYMENT (Cell 9) TO SEE ACTUAL RESULTS
# Check what's actually deployed vs what's in Git repo
import os

print("🔍 Comparing Git repository vs deployed items...")
print("⚠️  This fetches FRESH data from Fabric workspace\n")

# 1. Scan Git repository for Fabric items
print("📂 Items in Git repository:")
git_items = {}
for entry in os.listdir(repo_path):
    if os.path.isdir(os.path.join(repo_path, entry)) and '.' in entry:
        parts = entry.split('.')
        if len(parts) == 2:
            item_name, item_type = parts
            if item_type not in git_items:
                git_items[item_type] = []
            git_items[item_type].append(item_name)

for item_type, names in sorted(git_items.items()):
    print(f"  📦 {item_type} ({len(names)}): {', '.join(names)}")

# 2. Fetch FRESH deployed items from workspace
print(f"\n📊 Items deployed to workspace (FRESH FETCH):")
response = requests.get(
    f"https://api.fabric.microsoft.com/v1/workspaces/{target_workspace_id}/items",
    headers=headers
)

if response.status_code == 200:
    # Store fresh items for use by next cells
    items = response.json().get("value", [])
    globals()['items'] = items  # Update global variable with fresh data
    
    # Group by type
    deployed_items = {}
    for item in items:
        item_type = item.get('type', 'Unknown')
        if item_type not in deployed_items:
            deployed_items[item_type] = []
        deployed_items[item_type].append(item['displayName'])
    
    # Store for next cell
    globals()['deployed_items'] = deployed_items
    
    for item_type, names in sorted(deployed_items.items()):
        print(f"  📦 {item_type} ({len(names)}): {', '.join(names)}")
    
    # 3. Compare and show what's missing
    print(f"\n❓ Missing items (in Git but not deployed):")
    
    # Map Git folder extensions to Fabric item types
    type_mapping = {
        'Reflex': 'Reflex',
        'DataAgent': 'DataAgent', 
        'Map': 'Map',
        'Eventstream': 'Eventstream',
        'Ontology': 'Ontology',
        'Eventhouse': 'Eventhouse',
        'KQLDatabase': 'KQLDatabase',
        'Lakehouse': 'Lakehouse',
        'Notebook': 'Notebook',
        'SemanticModel': 'SemanticModel',
        'Report': 'Report'
    }
    
    missing_count = 0
    for git_type, git_names in git_items.items():
        fabric_type = type_mapping.get(git_type, git_type)
        deployed_names = deployed_items.get(fabric_type, [])
        
        for git_name in git_names:
            if git_name not in deployed_names:
                print(f"  ⚠️ {git_type}: {git_name}")
                missing_count += 1
    
    if missing_count == 0:
        print(f"  ✅ All items from Git are deployed!")
    
    # 4. Special check for KQL Database (child of Eventhouse)
    print(f"\n🔍 KQL Database check:")
    if 'KQLDatabase' in deployed_items:
        print(f"  ✅ KQL Database deployed: {', '.join(deployed_items['KQLDatabase'])}")
    elif 'Eventhouse' in deployed_items:
        print(f"  ⚠️  Eventhouse exists but KQL Database not showing up yet")
        print(f"     💡 KQL Database may still be initializing - wait a minute and re-run this cell")
    else:
        print(f"  ❌ No Eventhouse or KQL Database found")
    
    print(f"\n📊 Summary:")
    print(f"  Git items: {sum(len(v) for v in git_items.values())} across {len(git_items)} types")
    print(f"  Deployed: {len(items)} items across {len(deployed_items)} types")
    print(f"  Missing: {missing_count}")
    
    if missing_count == 0:
        print(f"\n🎉 SUCCESS! All items deployed correctly!")
    else:
        print(f"\n⚠️  Run cell 11 to retry deploying missing items")
    
else:
    print(f"❌ Failed to list deployed items: {response.status_code}")

🔍 Comparing Git repository vs deployed items...
⚠️  This fetches FRESH data from Fabric workspace

📂 Items in Git repository:
  📦 DataAgent (1): maritimeDA
  📦 Eventhouse (1): maritimeEH
  📦 Eventstream (1): maritimeES
  📦 Lakehouse (1): maritimeLH
  📦 Map (1): vessels_map
  📦 Notebook (4): createOntology, runVesselswithSimulation2, runVesselswithSimulation, runVessels
  📦 Ontology (1): maritimeOntologyfromSM
  📦 Reflex (1): RedAlertActivator
  📦 Report (1): Vessels By Company
  📦 SemanticModel (1): maritimeSM
  📦 git (1): 
  📦 vscode (1): 

📊 Items deployed to workspace (FRESH FETCH):
  📦 DataAgent (1): maritimeDA
  📦 Eventhouse (1): maritimeEH
  📦 Eventstream (1): maritimeES
  📦 GraphModel (1): maritimeOntologyfromSM_graph_3e9a5032f01a4651902f33e46e09e231
  📦 KQLDatabase (1): maritimeEH
  📦 Lakehouse (2): maritimeLH, maritimeOntologyfromSM_lh_3e9a5032f01a4651902f33e46e09e231
  📦 Map (1): vessels_map
  📦 Notebook (4): runVessels, runVesselswithSimulation2, runVesselswithSimulation, cr